# 01 - Data Exploration: EEG Eye State Dataset

Loads raw EEG recordings, applies preprocessing, and explores the alpha-band "eyes open vs. eyes closed" signal before any modeling happens.

**Dataset:** [PhysioNet EEG Motor Movement/Imagery Dataset](https://physionet.org/content/eegmmidb/1.0.0/) (Goldberger et al., 2000).
Each subject contributes two baseline runs: `R01` (eyes open, resting) and `R02` (eyes closed, resting).

**What this notebook does:**
1. Loads EDF files for a set of subjects
2. Bandpass-filters and windows the recordings
3. Extracts per-channel, per-band spectral power (delta, theta, alpha, beta) via Welch's method
4. Builds the dataset (`X`, `y`) shared by the KNN and Random Forest notebooks
5. Visualizes the alpha-band separation between eyes-open and eyes-closed windows, confirming the physiological signal exists before modeling on it

Run this notebook first — `02_knn_classifier.ipynb` and `03_random_forest_classifier.ipynb` both assume this same feature-extraction approach.


## Setup

In [ ]:
# In Colab, uncomment the install line below.
# !pip install mne scikit-learn matplotlib seaborn numpy scipy pandas

import os
import random
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from scipy.signal import welch

mne.set_log_level("WARNING")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)


## Data location

Two ways to get the EDF files:

**Option A - auto-download (recommended, works for anyone who clones this repo):**
```python
from mne.datasets import eegbci
files = eegbci.load_data(subject=1, runs=[1, 2], update_path=True)
```

**Option B - your own Drive folder** (what this notebook uses below, matching the original project setup):
```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/eeg_eyes_project"
```

Pick whichever matches your setup. The rest of the notebook just needs a folder of `S0##R01.edf` / `S0##R02.edf` files.

In [ ]:
# --- Option B: Google Drive (uncomment if running in Colab with data already in Drive) ---
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/eeg_eyes_project"

print(f"Files in {DATA_DIR} (first 10):")
print(os.listdir(DATA_DIR)[:10])


## Preprocessing and feature extraction

Bandpass filter -> 2-second non-overlapping windows -> Welch PSD -> band power per channel, across four canonical EEG bands.

In [ ]:
WINDOW_SEC = 2.0

WAVE_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
}

def bandpower(signal, sf, band):
    """Integrated power spectral density in `band`, via Welch's method."""
    fmin, fmax = band
    freqs, psd = welch(signal, sf, nperseg=min(256, len(signal)))
    idx = (freqs >= fmin) & (freqs <= fmax)
    return np.trapz(psd[idx], freqs[idx])

def extract_band_features(file_path, label, window_sec=WINDOW_SEC):
    """Load one EDF file, filter, window, and extract per-channel band power features."""
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    raw.pick_types(eeg=True)
    raw.set_eeg_reference("average", projection=False, verbose=False)
    raw.filter(1.0, 40.0, fir_design="firwin", verbose=False)

    data = raw.get_data()
    sfreq = raw.info["sfreq"]
    ch_names = raw.info["ch_names"]

    win_samples = int(window_sec * sfreq)
    n_windows = data.shape[1] // win_samples
    if n_windows == 0:
        return None, None, None

    data = data[:, : n_windows * win_samples]
    data = data.reshape(data.shape[0], n_windows, win_samples)
    windows = np.transpose(data, (1, 0, 2))  # (n_windows, n_channels, n_samples)

    n_bands = len(WAVE_BANDS)
    X = np.zeros((n_windows, data.shape[0] * n_bands))
    for i in range(n_windows):
        feats = []
        for band in WAVE_BANDS.values():
            for ch in range(windows.shape[1]):
                feats.append(bandpower(windows[i, ch], sfreq, band))
        X[i] = feats

    y = np.full(n_windows, label)
    return X, y, ch_names


def build_dataset(data_dir, window_sec=WINDOW_SEC):
    X_list, y_list = [], []
    ch_names = None

    files = sorted(glob(os.path.join(data_dir, "*.edf")))
    print(f"Found {len(files)} EDF files")

    for fpath in files:
        fname = os.path.basename(fpath)
        if "R01" in fname:
            label = 0  # eyes open
        elif "R02" in fname:
            label = 1  # eyes closed
        else:
            continue

        X_feats, y_feats, names = extract_band_features(fpath, label, window_sec)
        if X_feats is None:
            print(f"  Skipping {fname} (too short)")
            continue

        X_list.append(X_feats)
        y_list.append(y_feats)
        if ch_names is None:
            ch_names = names

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    return X, y, ch_names


In [ ]:
X, y, ch_names = build_dataset(DATA_DIR, window_sec=WINDOW_SEC)

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)
print("Class counts [open, closed]:", np.bincount(y))


In [ ]:
# Save the dataset so 02_knn_classifier.ipynb and 03_random_forest_classifier.ipynb can load it directly
# instead of re-running feature extraction from scratch.
np.savez("eeg_features.npz", X=X, y=y, ch_names=np.array(ch_names))
print("Saved eeg_features.npz")


## Sanity check: does alpha power actually separate the two classes?

Before trusting any model's accuracy, confirm the underlying physiological signal is visible in the raw features.

In [ ]:
n_bands = len(WAVE_BANDS)
n_channels = len(ch_names)
band_names = list(WAVE_BANDS.keys())
alpha_idx = band_names.index("alpha")

start = alpha_idx * n_channels
end = (alpha_idx + 1) * n_channels
alpha_feats = X[:, start:end]

open_alpha = alpha_feats[y == 0].mean(axis=0)
closed_alpha = alpha_feats[y == 1].mean(axis=0)

plt.figure(figsize=(10, 4))
plt.plot(open_alpha, marker="o", label="Eyes Open")
plt.plot(closed_alpha, marker="o", label="Eyes Closed")
plt.xlabel("Channel Index")
plt.ylabel("Mean Alpha Power (a.u.)")
plt.title("Average Alpha Power Across Channels")
plt.legend()
plt.tight_layout()
plt.savefig("alpha_power_by_channel.png", dpi=150)
plt.show()


If eyes-closed alpha power sits consistently above eyes-open across most channels (as expected from the alpha-blocking phenomenon), the feature set is doing its job and it's reasonable to move on to `02_knn_classifier.ipynb` and `03_random_forest_classifier.ipynb`.